# Day 4 — Exploratory Data Analysis: New York Taxi Rides
**Goal:** Turn the cleaned taxi dataset into a one-page data story for a business owner.



## 1. Import the libraries

These libraries help me load and summarize the data, create charts, and export the final PDF.

In [3]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10

ModuleNotFoundError: No module named 'matplotlib'

## 2. Load the cleaned data

`read_csv()` opens the CSV file. `head()` lets me preview the first rows, and `shape` tells me how many rows and columns are in the dataset.

In [ ]:
DATA_PATH = Path("cleaned.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "I could not find cleaned.csv. Put cleaned.csv in the same week1 folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

## 3. Prepare the columns used in the analysis

I convert the numeric columns to numbers in case a value was stored as text. I also extract the pickup hour from the pickup timestamp. This step does not delete any rides.

In [ ]:
numeric_cols = ["distance", "fare", "tip", "tolls", "total", "passengers"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

if "pickup" in df.columns:
    df["pickup"] = pd.to_datetime(df["pickup"], errors="coerce")
    df["pickup_hour"] = df["pickup"].dt.hour

required_cols = ["distance", "fare", "tip", "tolls", "total", "passengers"]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in cleaned.csv: {missing_cols}")

print(f"Rows available for analysis: {len(df):,}")
print("Missing values in selected numeric columns:")
display(df[required_cols].isna().sum().to_frame("missing_values"))

## 4. Chart 1 — How are ride distances spread?

**Why a histogram?** A histogram groups numeric distances into ranges. It helps me see which ride lengths happen most often and whether a few rides are much longer than the rest.

In [ ]:
distance_data = df["distance"].dropna()

fig1, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(distance_data, bins=30, edgecolor="white")
ax.set_title("Distribution of Taxi Ride Distances")
ax.set_xlabel("Distance")
ax.set_ylabel("Number of rides")
ax.text(
    0.01, -0.22,
    "Business meaning: Frequent short rides may represent the core demand; unusually long rides may need separate pricing or service planning.",
    transform=ax.transAxes, fontsize=9, wrap=True
)
fig1.tight_layout()
plt.show()

print("My interpretation:")
print("1. The histogram shows how frequently different ride-distance ranges occur.")
print("2. A business can use this pattern to understand its typical trip length and plan pricing or driver availability.")

## 5. Chart 2 — How much revenue comes from each pickup service zone?

**Why a bar chart?** Service zones are categories. Bars make it easy to compare the total revenue associated with each category.

In [ ]:
zone_col = "service_zone" if "service_zone" in df.columns else "pickup_zone"
zone_revenue = (
    df.groupby(zone_col, dropna=False)["total"]
      .sum()
      .sort_values(ascending=False)
)
# Keep the top 10 zones readable, and group the rest as Other zones.
top_n = 10
if len(zone_revenue) > top_n:
    plotted_revenue = zone_revenue.head(top_n).copy()
    plotted_revenue.loc["Other zones"] = zone_revenue.iloc[top_n:].sum()
else:
    plotted_revenue = zone_revenue

fig2, ax = plt.subplots(figsize=(8, 4.8))
plotted_revenue.sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Total Revenue by Pickup {zone_col.replace('_', ' ').title()}")
ax.set_xlabel("Total revenue")
ax.set_ylabel("Pickup service zone")
ax.text(
    0.01, -0.20,
    "Business meaning: High-revenue zones may deserve closer attention when planning driver supply and service coverage.",
    transform=ax.transAxes, fontsize=9, wrap=True
)
fig2.tight_layout()
plt.show()

print("My interpretation:")
print(f"1. This chart compares total recorded revenue across pickup categories using '{zone_col}'.")
print("2. The highest-revenue zones may be useful starting points for driver allocation, but revenue alone does not prove profit or demand per driver.")

## 6. Chart 3 — At what hours do pickups happen most often?

**Why a line chart?** Hours follow a natural order from 0 to 23. A line makes changes across the day easier to see.

In [ ]:
if "pickup_hour" not in df.columns:
    raise ValueError("The pickup timestamp could not be found, so pickup hours cannot be calculated.")

hour_counts = (
    df["pickup_hour"].dropna().astype(int)
      .value_counts()
      .reindex(range(24), fill_value=0)
      .sort_index()
)

fig3, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(hour_counts.index, hour_counts.values, marker="o", linewidth=2)
ax.set_title("Taxi Pickups by Hour of Day")
ax.set_xlabel("Pickup hour (24-hour clock)")
ax.set_ylabel("Number of rides")
ax.set_xticks(range(0, 24, 2))
ax.text(
    0.01, -0.22,
    "Business meaning: Hours with more pickups may need more available drivers; confirm this pattern with a larger or more recent dataset before changing schedules.",
    transform=ax.transAxes, fontsize=9, wrap=True
)
fig3.tight_layout()
plt.show()

busiest_hour = int(hour_counts.idxmax())
print("My interpretation:")
print(f"1. The busiest hour in this dataset is {busiest_hour:02d}:00, with {int(hour_counts.max()):,} pickups.")
print("2. The hourly pattern can help with shift planning, although this dataset alone may not represent every day or season.")

## 7. Chart 4 — How is fare related to ride distance?

**Why a scatter plot?** Each point represents a ride. It helps show whether fares tend to increase with distance and whether there are unusual rides.

In [ ]:
scatter_data = df[["distance", "fare"]].dropna()
fig4, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(scatter_data["distance"], scatter_data["fare"], alpha=0.35, s=18)
ax.set_title("Fare vs. Ride Distance")
ax.set_xlabel("Distance")
ax.set_ylabel("Fare")
ax.text(
    0.01, -0.22,
    "Business meaning: A general upward pattern would be consistent with longer rides costing more, while exceptions may reflect other fare components or unusual records.",
    transform=ax.transAxes, fontsize=9, wrap=True
)
fig4.tight_layout()
plt.show()

distance_fare_corr = scatter_data["distance"].corr(scatter_data["fare"])
print("My interpretation:")
print(f"1. The Pearson correlation between distance and fare is {distance_fare_corr:.2f}.")
print("2. Correlation describes how two values move together; it does not prove that distance alone causes the fare.")

## 8. Correlation matrix — Which numeric fields move together?

A correlation matrix shows the strength and direction of linear relationships between numeric columns. Values close to **+1** indicate a strong positive relationship, values close to **-1** indicate a strong negative relationship, and values near **0** indicate little linear relationship.

Correlation is not proof of cause and effect.

In [ ]:
corr_cols = ["distance", "fare", "tip", "tolls", "total", "passengers"]
corr_matrix = df[corr_cols].corr()

fig5, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, center=0, square=True, ax=ax
)
ax.set_title("Correlation Matrix of Ride and Payment Variables")
fig5.tight_layout()
plt.show()

# Find the strongest off-diagonal relationship so the interpretation is based on this dataset.
corr_pairs = corr_matrix.where(
    ~pd.DataFrame(
        [[i == j for j in corr_matrix.columns] for i in corr_matrix.columns],
        index=corr_matrix.index, columns=corr_matrix.columns
    )
).stack().sort_values(ascending=False)

# Each pair appears twice, so remove duplicate pair order.
unique_pairs = []
seen = set()
for (a, b), value in corr_pairs.items():
    key = tuple(sorted((a, b)))
    if key not in seen:
        seen.add(key)
        unique_pairs.append((a, b, value))
unique_pairs.sort(key=lambda item: abs(item[2]), reverse=True)

if unique_pairs:
    a, b, value = unique_pairs[0]
    direction = "positive" if value > 0 else "negative" if value < 0 else "near zero"
    print("My interpretation:")
    print(f"1. The strongest absolute correlation in this matrix is between {a} and {b} (r = {value:.2f}), a {direction} relationship.")
    print("2. Strong relationships can help explain how ride and payment fields relate, but business decisions should also consider context and data quality.")

## 9. Create the one-page PDF data story

The next cell exports a compact PDF containing all four charts and the correlation heatmap. Run all cells first. The PDF is saved as `day4_story.pdf` in the same folder as this notebook.

In [ ]:
# Create a compact, multi-panel one-page PDF using the figures generated above.
pdf_path = Path("day4_story.pdf")

# Create a single landscape page with 5 panels.
from matplotlib.gridspec import GridSpec

story_fig = plt.figure(figsize=(16, 11))
story_fig.suptitle(
    "NEW YORK TAXI RIDES | A ONE-PAGE DATA STORY",
    fontsize=19, fontweight="bold", y=0.98
)
story_fig.text(
    0.5, 0.945,
    f"Analysis of {len(df):,} cleaned rides | Exploratory findings, not causal claims",
    ha="center", fontsize=10
)

gs = GridSpec(3, 4, figure=story_fig, left=0.06, right=0.97, top=0.90, bottom=0.10,
              hspace=0.55, wspace=0.45)

# Recreate compact chart panels for the PDF.
ax1 = story_fig.add_subplot(gs[0, :2])
ax1.hist(distance_data, bins=25, edgecolor="white")
ax1.set_title("1. Ride distance distribution")
ax1.set_xlabel("Distance"); ax1.set_ylabel("Rides")

ax2 = story_fig.add_subplot(gs[0, 2:])
plotted_revenue.sort_values().plot(kind="barh", ax=ax2)
ax2.set_title("2. Revenue by pickup zone")
ax2.set_xlabel("Total revenue"); ax2.set_ylabel("")

ax3 = story_fig.add_subplot(gs[1, :2])
ax3.plot(hour_counts.index, hour_counts.values, marker="o", linewidth=1.8)
ax3.set_title("3. Pickups by hour")
ax3.set_xlabel("Hour (24-hour clock)"); ax3.set_ylabel("Rides")
ax3.set_xticks(range(0, 24, 4))

ax4 = story_fig.add_subplot(gs[1, 2:])
ax4.scatter(scatter_data["distance"], scatter_data["fare"], alpha=0.3, s=10)
ax4.set_title("4. Fare vs. distance")
ax4.set_xlabel("Distance"); ax4.set_ylabel("Fare")

ax5 = story_fig.add_subplot(gs[2, :2])
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            center=0, square=True, cbar=False, annot_kws={"size": 8}, ax=ax5)
ax5.set_title("5. Correlation matrix")

# Short, dataset-grounded story notes on the right.
axnote = story_fig.add_subplot(gs[2, 2:])
axnote.axis("off")
axnote.set_title("WHAT THIS MEANS FOR THE BUSINESS", loc="left", fontsize=12, pad=10)
note_text = (
    "• Distance: Use the distribution to understand common trip lengths and identify long trips.\n\n"
    f"• Zones: Highest revenue category shown is {zone_revenue.index[0]} "
    f"({zone_revenue.iloc[0]:,.2f} in recorded total revenue). Revenue is not the same as profit.\n\n"
    f"• Timing: Busiest pickup hour is {busiest_hour:02d}:00 "
    f"({int(hour_counts.max()):,} rides). Consider it as a scheduling clue, not a guarantee.\n\n"
    f"• Fare and distance: correlation = {distance_fare_corr:.2f}. Other factors may affect fares.\n\n"
    "• Correlation: Relationships help generate questions; they do not establish cause and effect."
)
axnote.text(0, 0.98, note_text, va="top", fontsize=9.5, wrap=True)

story_fig.text(
    0.06, 0.035,
    "Data source: Day 3 cleaned.csv. Interpretations are descriptive and should be validated against broader, current data.",
    fontsize=8
)

story_fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
plt.show()
print(f"PDF created: {pdf_path.resolve()}")